# Domain Adaptation for LLMs

## Overview

Domain adaptation specializes general-purpose language models for specific domains like medicine, law, code, finance, or science.

### Why Domain Adaptation?

General LLMs (GPT, Llama) are trained on diverse web data:
- **Broad knowledge** but shallow expertise
- **Generic language** not domain-specific terminology
- **Lack specialized reasoning** patterns

Domain-specific applications need:
- **Deep expertise** in specialized topics
- **Technical vocabulary** and jargon
- **Domain conventions** and reasoning patterns
- **High accuracy** on specialized tasks

### Historical Examples

- **BioBERT** (2019): BERT for biomedical text
- **ClinicalBERT** (2019): Clinical notes understanding
- **CodeBERT** (2020): Programming language understanding
- **BloombergGPT** (2023): 50B param model for finance
- **Med-PaLM** (2023): Medical question answering
- **GPT-4 Code** (2023): Specialized for programming

### Key Challenge

**Catastrophic forgetting**: Fine-tuning on domain data can degrade general capabilities

Must balance:
- Domain expertise
- General knowledge retention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from tqdm import tqdm
import math

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Domain-Specific Challenges

### Medical Domain

**Challenges**:
- Complex medical terminology ("hepatosplenomegaly")
- Abbreviations and acronyms ("HTN", "DM", "CAD")
- Clinical reasoning patterns
- Patient safety critical
- Regulatory compliance (HIPAA)

**Data sources**:
- PubMed abstracts
- Clinical notes (de-identified)
- Medical textbooks
- Drug databases

### Legal Domain

**Challenges**:
- Legal terminology ("habeas corpus", "tort")
- Citation formats
- Precedent-based reasoning
- Jurisdiction-specific laws
- High accuracy requirements

**Data sources**:
- Legal documents and contracts
- Court opinions
- Legal textbooks
- Statutes and regulations

### Code Domain

**Challenges**:
- Multiple programming languages
- API knowledge
- Algorithmic reasoning
- Code structure and style
- Bug detection

**Data sources**:
- GitHub repositories
- Stack Overflow
- Documentation
- Competitive programming

### Finance Domain

**Challenges**:
- Financial terminology
- Numerical reasoning
- Market dynamics
- Risk assessment
- Real-time data

**Data sources**:
- Financial news
- SEC filings
- Earnings calls
- Market data

In [ ]:
# Visualize domain characteristics
def visualize_domain_characteristics():
    """Compare characteristics across domains."""
    
    domains = ['General', 'Medical', 'Legal', 'Code', 'Finance']
    
    characteristics = {
        'Specialized\nVocabulary': [3, 9, 8, 7, 7],
        'Technical\nDepth': [4, 9, 8, 9, 7],
        'Reasoning\nComplexity': [5, 8, 9, 8, 7],
        'Accuracy\nRequirement': [6, 10, 10, 8, 9],
        'Data\nAvailability': [10, 6, 5, 9, 7],
    }
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(domains))
    width = 0.15
    multiplier = 0
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    
    for (characteristic, values), color in zip(characteristics.items(), colors):
        offset = width * multiplier
        ax.bar(x + offset, values, width, label=characteristic, color=color, alpha=0.8)
        multiplier += 1
    
    ax.set_xlabel('Domain', fontsize=12)
    ax.set_ylabel('Intensity (0-10)', fontsize=12)
    ax.set_title('Domain-Specific Challenges', fontsize=14, weight='bold')
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(domains)
    ax.legend(loc='upper left', ncol=2, fontsize=10)
    ax.set_ylim(0, 11)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Key Observations ===")
    print("\nMedical & Legal: Highest accuracy requirements")
    print("Code: Best data availability (GitHub)")
    print("All specialized domains: Need technical vocabulary")
    print("Challenge: Adapt while maintaining general capabilities")

visualize_domain_characteristics()

## 2. Continued Pre-training

### Concept

Continue pre-training on domain-specific data using the same objective (language modeling).

```
General LM → Continued Pre-training on Domain Data → Domain LM
```

### Procedure

1. **Start**: Load general pre-trained model
2. **Data**: Collect large domain corpus
3. **Train**: Continue language modeling objective
4. **Result**: Model learns domain vocabulary and patterns

### Advantages

- **Simple**: Same objective as pre-training
- **Effective**: Learns domain knowledge naturally
- **Flexible**: Can use unlabeled data

### Challenges

- **Forgetting**: May lose general capabilities
- **Data volume**: Needs substantial domain corpus
- **Compute**: Expensive for large models

### Hyperparameters

- **Learning rate**: Lower than original pre-training (1e-5 to 1e-4)
- **Warmup**: Gradual learning rate increase
- **Steps**: 5-20% of original pre-training steps
- **Batch size**: Same as pre-training

In [ ]:
@dataclass
class DomainCorpus:
    """Domain-specific text corpus."""
    domain: str
    texts: List[str]
    size_mb: float
    
    def __len__(self):
        return len(self.texts)


def create_domain_corpus(domain: str, num_docs: int = 1000) -> DomainCorpus:
    """Create synthetic domain corpus."""
    
    templates = {
        'medical': [
            "Patient presents with {} and {}. Physical examination reveals {}.",
            "Diagnosis: {}. Treatment plan includes {} and {}.",
            "The pathophysiology of {} involves {}.",
        ],
        'legal': [
            "In the case of {} v. {}, the court held that {}.",
            "According to {} statute {}, {}.",
            "The plaintiff alleges {} and seeks {}.",
        ],
        'code': [
            "def {}(): # {}\n    return {}",
            "class {}:\n    def __init__(self): {}\n    def {}(self): {}",
            "# {} implementation using {}",
        ],
        'finance': [
            "The company's {} increased by {}% to ${}M.",
            "Market analysis shows {} with {} outlook.",
            "Risk factors include {} and {}.",
        ],
    }
    
    texts = []
    for _ in range(num_docs):
        template = np.random.choice(templates[domain])
        # Fill with placeholder terms
        filled = template.format(*[f'[{domain}_term_{i}]' for i in range(template.count('{}'))])
        texts.append(filled)
    
    # Estimate size
    size_mb = sum(len(t) for t in texts) / 1024 / 1024
    
    return DomainCorpus(domain, texts, size_mb)


class ContinuedPreTrainer:
    """Continued pre-training for domain adaptation."""
    
    def __init__(
        self,
        model: nn.Module,
        learning_rate: float = 1e-5,
        warmup_steps: int = 500,
        max_steps: int = 10000
    ):
        self.model = model
        self.learning_rate = learning_rate
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            betas=(0.9, 0.95),
            weight_decay=0.1
        )
    
    def get_lr(self, step: int) -> float:
        """Learning rate schedule with warmup."""
        if step < self.warmup_steps:
            return self.learning_rate * step / self.warmup_steps
        else:
            # Cosine decay
            progress = (step - self.warmup_steps) / (self.max_steps - self.warmup_steps)
            return self.learning_rate * 0.5 * (1 + math.cos(math.pi * progress))
    
    def train_step(
        self,
        input_ids: torch.Tensor,
        step: int
    ) -> float:
        """Single training step."""
        self.model.train()
        
        # Update learning rate
        lr = self.get_lr(step)
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        
        # Forward pass (language modeling)
        logits = self.model(input_ids)
        
        # Shift for next-token prediction
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        
        # Loss
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )
        
        # Backward pass
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        
        return loss.item()


# Demo corpus creation
print("=== Domain Corpus Examples ===")
for domain in ['medical', 'legal', 'code', 'finance']:
    corpus = create_domain_corpus(domain, num_docs=100)
    print(f"\n{domain.capitalize()} Corpus:")
    print(f"  Documents: {len(corpus)}")
    print(f"  Size: {corpus.size_mb:.2f} MB")
    print(f"  Sample: {corpus.texts[0][:100]}...")

## 3. Domain-Specific Fine-tuning

### Supervised Fine-tuning on Domain Tasks

After continued pre-training, fine-tune on task-specific data:

```
General LM → Continued Pre-training → Domain SFT → Task Model
```

### Task Types

**Medical**:
- Question answering (medical QA)
- Clinical note generation
- Diagnosis prediction
- Drug interaction detection

**Legal**:
- Contract analysis
- Legal research
- Document summarization
- Compliance checking

**Code**:
- Code generation
- Bug fixing
- Code explanation
- Test generation

**Finance**:
- Financial analysis
- Risk assessment
- Report generation
- Market prediction

### Data Requirements

- **Quality over quantity**: High-quality labeled examples
- **Diversity**: Cover various sub-tasks
- **Format**: Instruction-response pairs
- **Size**: 1K-100K examples depending on task

In [ ]:
@dataclass
class DomainExample:
    """Domain-specific training example."""
    domain: str
    task: str
    instruction: str
    input: str
    output: str


def create_domain_examples(domain: str, num_examples: int = 50) -> List[DomainExample]:
    """Create synthetic domain-specific examples."""
    
    examples = []
    
    if domain == 'medical':
        tasks = ['diagnosis', 'treatment', 'explanation']
        for i in range(num_examples):
            task = np.random.choice(tasks)
            if task == 'diagnosis':
                instruction = "Based on the symptoms, suggest possible diagnoses."
                input_text = f"Patient {i}: fever, cough, fatigue"
                output = f"Possible diagnoses: [medical_diagnosis_{i}]"
            elif task == 'treatment':
                instruction = "Recommend treatment for the condition."
                input_text = f"Condition: [medical_condition_{i}]"
                output = f"Treatment: [medical_treatment_{i}]"
            else:
                instruction = "Explain the medical term."
                input_text = f"Term: [medical_term_{i}]"
                output = f"Explanation: [medical_explanation_{i}]"
            
            examples.append(DomainExample(domain, task, instruction, input_text, output))
    
    elif domain == 'code':
        tasks = ['generate', 'explain', 'debug']
        for i in range(num_examples):
            task = np.random.choice(tasks)
            if task == 'generate':
                instruction = "Write Python code for the task."
                input_text = f"Task: [code_task_{i}]"
                output = f"def solution():\n    # [code_{i}]\n    pass"
            elif task == 'explain':
                instruction = "Explain what this code does."
                input_text = f"Code: [code_snippet_{i}]"
                output = f"Explanation: [code_explanation_{i}]"
            else:
                instruction = "Find and fix the bug."
                input_text = f"Buggy code: [buggy_code_{i}]"
                output = f"Fixed code: [fixed_code_{i}]"
            
            examples.append(DomainExample(domain, task, instruction, input_text, output))
    
    return examples


class DomainDataset(Dataset):
    """Dataset for domain-specific fine-tuning."""
    
    def __init__(self, examples: List[DomainExample], max_length: int = 512):
        self.examples = examples
        self.max_length = max_length
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        example = self.examples[idx]
        
        # Format as instruction-following
        text = f"""### Instruction:
{example.instruction}

### Input:
{example.input}

### Response:
{example.output}"""
        
        # Simple tokenization (in practice, use proper tokenizer)
        tokens = [ord(c) % 1000 for c in text[:self.max_length]]
        
        return {
            'input_ids': torch.tensor(tokens),
            'attention_mask': torch.ones(len(tokens))
        }


# Demo domain examples
print("=== Domain-Specific Examples ===")

for domain in ['medical', 'code']:
    examples = create_domain_examples(domain, num_examples=3)
    
    print(f"\n{domain.upper()} Domain:")
    for i, ex in enumerate(examples, 1):
        print(f"\nExample {i} ({ex.task}):")
        print(f"  Instruction: {ex.instruction}")
        print(f"  Input: {ex.input[:60]}...")
        print(f"  Output: {ex.output[:60]}...")

## 4. Catastrophic Forgetting Prevention

### The Problem

Fine-tuning on domain data can cause model to:
- Forget general knowledge
- Lose capability on other tasks
- Overfit to domain specifics

### Metrics

Track both:
- **Domain performance**: Accuracy on domain tasks ↑
- **General performance**: Accuracy on general tasks ↓

### Prevention Strategies

#### 1. Experience Replay

Mix domain data with general data:
```python
batch = {
    'domain_data': 80%,
    'general_data': 20%,
}
```

#### 2. Elastic Weight Consolidation (EWC)

Penalize changes to important parameters:
$$L_{EWC} = L_{domain} + \lambda \sum_i F_i (\theta_i - \theta_i^*)^2$$

where $F_i$ is Fisher information (parameter importance).

#### 3. Progressive Neural Networks

Add new parameters for domain, keep old frozen:
```
General Model (frozen) + Domain Adapter (trainable)
```

#### 4. LoRA for Domain Adaptation

Train low-rank adapters instead of full model:
```
W_adapted = W_frozen + α · (B × A)
```

#### 5. Multi-task Learning

Train on domain + general tasks simultaneously:
```python
loss = domain_loss + λ · general_loss
```

#### 6. Lower Learning Rate

Use smaller LR to make gradual changes:
- Continued pre-training: 1e-5
- Domain fine-tuning: 1e-6 to 1e-5

In [ ]:
def simulate_catastrophic_forgetting():
    """Simulate forgetting during domain adaptation."""
    
    steps = np.arange(0, 1001, 100)
    
    # Scenario 1: No forgetting prevention
    domain_perf_no_prevention = 0.5 + 0.4 * (1 - np.exp(-0.003 * steps))
    general_perf_no_prevention = 0.8 - 0.4 * (1 - np.exp(-0.002 * steps))
    
    # Scenario 2: With experience replay
    domain_perf_replay = 0.5 + 0.35 * (1 - np.exp(-0.003 * steps))
    general_perf_replay = 0.8 - 0.15 * (1 - np.exp(-0.002 * steps))
    
    # Scenario 3: With LoRA (best)
    domain_perf_lora = 0.5 + 0.38 * (1 - np.exp(-0.003 * steps))
    general_perf_lora = 0.8 - 0.05 * (1 - np.exp(-0.002 * steps))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Domain performance
    axes[0].plot(steps, domain_perf_no_prevention, 'o-', label='No prevention', linewidth=2)
    axes[0].plot(steps, domain_perf_replay, 's-', label='Experience replay', linewidth=2)
    axes[0].plot(steps, domain_perf_lora, '^-', label='LoRA', linewidth=2)
    axes[0].set_xlabel('Training Steps', fontsize=12)
    axes[0].set_ylabel('Domain Task Accuracy', fontsize=12)
    axes[0].set_title('Domain Performance', fontsize=14)
    axes[0].legend()
    axes[0].set_ylim(0.4, 1.0)
    axes[0].grid(True, alpha=0.3)
    
    # General performance
    axes[1].plot(steps, general_perf_no_prevention, 'o-', label='No prevention', linewidth=2)
    axes[1].plot(steps, general_perf_replay, 's-', label='Experience replay', linewidth=2)
    axes[1].plot(steps, general_perf_lora, '^-', label='LoRA', linewidth=2)
    axes[1].set_xlabel('Training Steps', fontsize=12)
    axes[1].set_ylabel('General Task Accuracy', fontsize=12)
    axes[1].set_title('General Performance (Forgetting)', fontsize=14)
    axes[1].legend()
    axes[1].set_ylim(0.4, 1.0)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Catastrophic Forgetting Analysis ===")
    print("\nNo prevention:")
    print(f"  Domain: {domain_perf_no_prevention[0]:.2f} → {domain_perf_no_prevention[-1]:.2f} (+{domain_perf_no_prevention[-1]-domain_perf_no_prevention[0]:.2f})")
    print(f"  General: {general_perf_no_prevention[0]:.2f} → {general_perf_no_prevention[-1]:.2f} ({general_perf_no_prevention[-1]-general_perf_no_prevention[0]:.2f})")
    
    print("\nExperience replay:")
    print(f"  Domain: {domain_perf_replay[0]:.2f} → {domain_perf_replay[-1]:.2f} (+{domain_perf_replay[-1]-domain_perf_replay[0]:.2f})")
    print(f"  General: {general_perf_replay[0]:.2f} → {general_perf_replay[-1]:.2f} ({general_perf_replay[-1]-general_perf_replay[0]:.2f})")
    
    print("\nLoRA (best):")
    print(f"  Domain: {domain_perf_lora[0]:.2f} → {domain_perf_lora[-1]:.2f} (+{domain_perf_lora[-1]-domain_perf_lora[0]:.2f})")
    print(f"  General: {general_perf_lora[0]:.2f} → {general_perf_lora[-1]:.2f} ({general_perf_lora[-1]-general_perf_lora[0]:.2f})")
    
    print("\nKey finding: LoRA maintains general performance while gaining domain expertise!")

simulate_catastrophic_forgetting()

## 5. Data Mixing Strategies

### Why Mix Data?

Balance domain specialization with general capabilities.

### Mixing Approaches

#### 1. Fixed Ratio Mixing

```python
batch = {
    'domain': 0.8,
    'general': 0.2,
}
```

**Pros**: Simple, predictable
**Cons**: May not be optimal ratio

#### 2. Curriculum Mixing

Start general, gradually increase domain:
```python
domain_ratio(step) = min(0.9, 0.5 + 0.4 * step / max_steps)
```

**Pros**: Smooth transition
**Cons**: Requires tuning schedule

#### 3. Temperature-based Sampling

Sample with temperature:
```python
p_domain = exp(w_domain / T) / Z
p_general = exp(w_general / T) / Z
```

**Pros**: Probabilistic, flexible
**Cons**: Complex to tune

#### 4. Task-balanced Sampling

Equal probability per task:
```python
for task in tasks:
    sample_from(task, n / len(tasks))
```

**Pros**: Fair to all tasks
**Cons**: Ignores task difficulty

### Recommended Strategy

For domain adaptation:
```python
# Phase 1: Continued pre-training
domain: 95%, general: 5%

# Phase 2: Domain fine-tuning
domain: 80%, general: 20%

# Phase 3: Alignment
domain: 60%, general: 40%
```

In [ ]:
class DataMixer:
    """Utility for mixing domain and general data."""
    
    def __init__(
        self,
        domain_dataset: Dataset,
        general_dataset: Dataset,
        mixing_strategy: str = 'fixed',
        domain_ratio: float = 0.8
    ):
        self.domain_dataset = domain_dataset
        self.general_dataset = general_dataset
        self.mixing_strategy = mixing_strategy
        self.domain_ratio = domain_ratio
        self.step = 0
    
    def get_ratio(self, step: Optional[int] = None) -> float:
        """Get current domain ratio."""
        if step is None:
            step = self.step
        
        if self.mixing_strategy == 'fixed':
            return self.domain_ratio
        
        elif self.mixing_strategy == 'curriculum':
            # Start 50%, gradually increase to 90%
            progress = min(1.0, step / 10000)
            return 0.5 + 0.4 * progress
        
        elif self.mixing_strategy == 'reverse_curriculum':
            # Start 90%, gradually decrease to 70%
            progress = min(1.0, step / 10000)
            return 0.9 - 0.2 * progress
        
        return self.domain_ratio
    
    def sample_batch(self, batch_size: int) -> Dict[str, torch.Tensor]:
        """Sample mixed batch."""
        ratio = self.get_ratio()
        
        # Number of domain examples
        n_domain = int(batch_size * ratio)
        n_general = batch_size - n_domain
        
        # Sample indices
        domain_indices = np.random.choice(len(self.domain_dataset), n_domain)
        general_indices = np.random.choice(len(self.general_dataset), n_general)
        
        self.step += 1
        
        return {
            'domain_examples': n_domain,
            'general_examples': n_general,
            'ratio': ratio
        }


# Visualize mixing strategies
def visualize_mixing_strategies():
    """Compare different data mixing strategies."""
    
    steps = np.arange(0, 10001, 100)
    
    # Different strategies
    fixed = np.ones_like(steps) * 0.8
    curriculum = np.clip(0.5 + 0.4 * steps / 10000, 0, 1)
    reverse = np.clip(0.9 - 0.2 * steps / 10000, 0, 1)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(steps, fixed, '-', linewidth=2, label='Fixed (80%)', color='blue')
    ax.plot(steps, curriculum, '-', linewidth=2, label='Curriculum (50%→90%)', color='green')
    ax.plot(steps, reverse, '-', linewidth=2, label='Reverse (90%→70%)', color='orange')
    
    ax.set_xlabel('Training Step', fontsize=12)
    ax.set_ylabel('Domain Data Ratio', fontsize=12)
    ax.set_title('Data Mixing Strategies', fontsize=14, weight='bold')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # Add phase annotations
    ax.axvline(x=3333, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=6666, color='gray', linestyle='--', alpha=0.5)
    ax.text(1666, 0.95, 'Pre-training', ha='center', fontsize=10, style='italic')
    ax.text(5000, 0.95, 'Fine-tuning', ha='center', fontsize=10, style='italic')
    ax.text(8333, 0.95, 'Alignment', ha='center', fontsize=10, style='italic')
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Mixing Strategy Recommendations ===")
    print("\nFixed: Good default, simple to implement")
    print("Curriculum: Start general, gradually specialize")
    print("  - Good for complex domains")
    print("  - Prevents early overfitting")
    print("\nReverse: Start specialized, add general")
    print("  - Good for maintaining general skills")
    print("  - Used in final alignment phase")

visualize_mixing_strategies()

## 6. Evaluation on Domain Tasks

### Domain-Specific Benchmarks

#### Medical
- **MedQA**: Medical exam questions (USMLE)
- **PubMedQA**: PubMed abstract QA
- **MedMCQA**: Indian medical entrance exams
- **MMLU Medical**: Medical subset of MMLU

#### Legal
- **LegalBench**: Legal reasoning tasks
- **CaseHOLD**: Legal precedent identification
- **ContractNLI**: Contract understanding

#### Code
- **HumanEval**: Python code generation
- **MBPP**: Python programming problems
- **CodeXGLUE**: Code understanding benchmark
- **APPS**: Algorithmic programming problems

#### Finance
- **FinQA**: Financial numerical reasoning
- **ConvFinQA**: Conversational finance QA
- **FiQA**: Financial opinion mining

### Evaluation Metrics

**Accuracy metrics**:
- Exact match
- F1 score
- BLEU/ROUGE (generation)

**Domain-specific**:
- Medical: Diagnosis accuracy, safety
- Legal: Citation accuracy, precedent relevance
- Code: Pass@k, execution accuracy
- Finance: Numerical accuracy, reasoning steps

### General Capability Retention

Also evaluate on general benchmarks:
- **MMLU**: General knowledge
- **HellaSwag**: Common sense reasoning
- **TruthfulQA**: Truthfulness
- **GSM8K**: Math reasoning

In [ ]:
def simulate_domain_evaluation():
    """Simulate evaluation across domains and capabilities."""
    
    models = ['Base Model', 'Domain Adapted', 'Domain + Forgetting Prevention']
    
    # Performance scores (0-100)
    scores = {
        'Domain Tasks': [45, 82, 78],
        'General Tasks': [75, 55, 72],
        'Safety': [70, 65, 72],
        'Truthfulness': [68, 60, 70],
    }
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(models))
    width = 0.2
    multiplier = 0
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    for (category, values), color in zip(scores.items(), colors):
        offset = width * multiplier
        bars = ax.bar(x + offset, values, width, label=category, color=color, alpha=0.8)
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{height:.0f}', ha='center', va='bottom', fontsize=9)
        
        multiplier += 1
    
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Domain Adaptation Evaluation Results', fontsize=14, weight='bold')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(models)
    ax.legend(loc='upper left', ncol=2, fontsize=10)
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Evaluation Summary ===")
    print("\nBase Model:")
    print("  ✓ Strong general capabilities")
    print("  ✗ Weak domain performance")
    
    print("\nDomain Adapted (naive):")
    print("  ✓ Excellent domain performance (+37 pts)")
    print("  ✗ Significant forgetting (-20 pts general)")
    print("  ✗ Reduced safety and truthfulness")
    
    print("\nDomain + Forgetting Prevention:")
    print("  ✓ Strong domain performance (+33 pts)")
    print("  ✓ Maintains general capabilities (-3 pts only)")
    print("  ✓ Preserves safety and truthfulness")
    print("  → Best trade-off!")

simulate_domain_evaluation()

## 7. Best Practices and Recommendations

### Domain Adaptation Pipeline

```
1. Data Collection
   ├── Domain corpus (10-100GB)
   ├── Task-specific labeled data (1K-100K)
   └── General data for mixing (10-50GB)

2. Continued Pre-training
   ├── Mix: 95% domain, 5% general
   ├── Steps: 5-20% of original pre-training
   ├── LR: 1e-5 with warmup
   └── Monitor: Perplexity on both domains

3. Domain Fine-tuning
   ├── Mix: 80% domain tasks, 20% general tasks
   ├── Epochs: 3-5
   ├── LR: 1e-6 to 5e-6
   └── Technique: LoRA or full fine-tuning

4. Alignment (optional)
   ├── Mix: 60% domain, 40% general
   ├── Method: DPO or RLHF
   └── Focus: Safety and helpfulness

5. Evaluation
   ├── Domain benchmarks
   ├── General benchmarks
   ├── Safety evaluations
   └── Human evaluation
```

### Key Recommendations

#### Data Quality
1. **Curate carefully**: Domain quality > quantity
2. **Diverse sources**: Multiple data types and formats
3. **Clean data**: Remove duplicates and low-quality content
4. **Validation set**: Hold out domain test set

#### Training Strategy
1. **Use LoRA**: Prevents forgetting, faster training
2. **Mix data**: Always include some general data
3. **Monitor both**: Track domain AND general performance
4. **Early stopping**: Stop if general performance drops too much

#### Hyperparameters
1. **Lower LR**: 10x lower than pre-training
2. **Longer warmup**: Gradual adaptation
3. **Smaller batches**: Better for fine-tuning
4. **Gradient clipping**: Prevent instability

#### Evaluation
1. **Multiple metrics**: Not just accuracy
2. **Human evaluation**: Essential for production
3. **Edge cases**: Test on difficult examples
4. **Safety testing**: Domain-specific risks

### Common Pitfalls

1. **Overfitting**: Too many epochs on small dataset
2. **Forgetting**: No general data mixing
3. **Data leakage**: Test data in training
4. **Insufficient evaluation**: Only domain metrics
5. **Premature deployment**: Not enough testing

In [ ]:
# Summary visualization
def create_domain_adaptation_summary():
    """Create comprehensive summary visualization."""
    
    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    # 1. Pipeline stages
    ax1 = fig.add_subplot(gs[0, :])
    stages = ['Base Model', 'Continued\nPre-training', 'Domain\nFine-tuning', 'Alignment', 'Domain\nExpert']
    improvements = [0, 20, 35, 42, 45]
    ax1.plot(improvements, 'o-', linewidth=3, markersize=10, color='green')
    ax1.set_xticks(range(len(stages)))
    ax1.set_xticklabels(stages)
    ax1.set_ylabel('Domain Performance', fontsize=11)
    ax1.set_title('Domain Adaptation Pipeline', fontsize=13, weight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 50)
    
    # 2. Data mixing
    ax2 = fig.add_subplot(gs[1, 0])
    phases = ['Continued\nPre-train', 'Domain\nFine-tune', 'Alignment']
    domain_ratios = [0.95, 0.80, 0.60]
    general_ratios = [0.05, 0.20, 0.40]
    x = np.arange(len(phases))
    ax2.bar(x, domain_ratios, label='Domain', color='blue', alpha=0.7)
    ax2.bar(x, general_ratios, bottom=domain_ratios, label='General', color='orange', alpha=0.7)
    ax2.set_xticks(x)
    ax2.set_xticklabels(phases, fontsize=9)
    ax2.set_ylabel('Data Ratio', fontsize=10)
    ax2.set_title('Data Mixing Strategy', fontsize=12, weight='bold')
    ax2.legend(fontsize=9)
    ax2.set_ylim(0, 1)
    
    # 3. Performance comparison
    ax3 = fig.add_subplot(gs[1, 1])
    methods = ['Full\nFT', 'LoRA', 'Experience\nReplay']
    domain_perf = [85, 82, 80]
    general_perf = [60, 75, 70]
    x = np.arange(len(methods))
    width = 0.35
    ax3.bar(x - width/2, domain_perf, width, label='Domain', color='green', alpha=0.7)
    ax3.bar(x + width/2, general_perf, width, label='General', color='purple', alpha=0.7)
    ax3.set_xticks(x)
    ax3.set_xticklabels(methods, fontsize=9)
    ax3.set_ylabel('Accuracy', fontsize=10)
    ax3.set_title('Method Comparison', fontsize=12, weight='bold')
    ax3.legend(fontsize=9)
    ax3.set_ylim(0, 100)
    
    # 4. Success factors
    ax4 = fig.add_subplot(gs[2, :])
    factors = ['Data\nQuality', 'Mixing\nStrategy', 'Forgetting\nPrevention', 
               'Learning\nRate', 'Evaluation']
    importance = [9, 8, 9, 7, 8]
    colors_bars = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    bars = ax4.barh(factors, importance, color=colors_bars, alpha=0.7)
    ax4.set_xlabel('Importance', fontsize=11)
    ax4.set_title('Success Factors for Domain Adaptation', fontsize=13, weight='bold')
    ax4.set_xlim(0, 10)
    ax4.grid(True, alpha=0.3, axis='x')
    
    # Add values on bars
    for bar in bars:
        width = bar.get_width()
        ax4.text(width + 0.2, bar.get_y() + bar.get_height()/2,
                f'{width:.0f}/10', ha='left', va='center', fontsize=10, weight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*60)
    print("DOMAIN ADAPTATION SUMMARY")
    print("="*60)
    print("\nKey Takeaways:")
    print("\n1. Multi-phase approach: Pre-train → Fine-tune → Align")
    print("2. Always mix general data to prevent forgetting")
    print("3. LoRA provides best balance of performance and retention")
    print("4. Quality > Quantity for domain data")
    print("5. Comprehensive evaluation on both domains")
    print("\n" + "="*60)

create_domain_adaptation_summary()

## Summary

### Key Takeaways

1. **Domain adaptation** specializes general LLMs for specific fields (medical, legal, code, finance)

2. **Two-phase approach**:
   - Continued pre-training on domain corpus
   - Domain-specific fine-tuning on tasks

3. **Catastrophic forgetting** is the main challenge - model loses general capabilities

4. **Prevention strategies**:
   - Data mixing (80% domain, 20% general)
   - LoRA (parameter-efficient adaptation)
   - Experience replay
   - Lower learning rates

5. **Data quality** is more important than quantity

6. **Evaluation** must cover both domain expertise AND general capabilities

7. **Best practice**: LoRA + data mixing + comprehensive evaluation

### Real-World Examples

- **Med-PaLM 2**: Medical question answering, 85%+ on USMLE
- **BloombergGPT**: 50B finance model, outperforms GPT on finance tasks
- **CodeLlama**: Code-specialized Llama 2, 48% on HumanEval
- **Harvey**: Legal AI assistant used by law firms

### When to Domain Adapt

**Yes, if**:
- Significant domain-specific vocabulary
- High accuracy requirements
- Sufficient domain data (10GB+)
- General models underperform significantly

**No, if**:
- Domain is well-covered in general training
- Limited domain data available
- Few-shot prompting works well
- Cost/benefit doesn't justify

### Future Directions

1. **Multi-domain models**: Single model, multiple domains
2. **Modular adaptation**: Plug-and-play domain modules
3. **Lifelong learning**: Continuous adaptation without forgetting
4. **Automated data curation**: AI-assisted domain data collection
5. **Better evaluation**: Domain-specific benchmarks

### References

1. Lee et al. (2020): "BioBERT: a pre-trained biomedical language representation model"
2. Wu et al. (2023): "BloombergGPT: A Large Language Model for Finance"
3. Singhal et al. (2023): "Large language models encode clinical knowledge"
4. Roziere et al. (2023): "Code Llama: Open Foundation Models for Code"
5. Kirkpatrick et al. (2017): "Overcoming catastrophic forgetting in neural networks"
6. Hu et al. (2021): "LoRA: Low-Rank Adaptation of Large Language Models"

---

## End of Stage 4: Advanced Fine-tuning

You have completed notebooks 40-44 covering:
- Reward model training for RLHF
- PPO algorithm and implementation
- DPO as simpler alternative
- Constitutional AI for safety
- Domain adaptation techniques

These represent state-of-the-art methods for aligning and specializing language models!